# 02 — Seq2seq and Recurrent Attention

    **Companion chapter:** `02-seq2seq-and-recurrent-attention.md`

    ## Learning goals

    - Contrast a fixed context vector with a dynamic context vector.
- Implement Luong dot-product attention.
- Implement Bahdanau additive attention.
- Visualize source–target alignments.
- Trace one attentive decoder step.

    ## How to use this notebook

    Run the cells from top to bottom. Read the comments, change small values, and
    rerun the cell. Every notebook ends with practice prompts that can become
    GitHub issues, exercises, or discussion questions.

In [1]:
from __future__ import annotations

import math
import random
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


## 1. The fixed-vector bottleneck

A basic seq2seq encoder gives the decoder only the final encoder state.
Attention instead constructs a different weighted context for each decoder step.

In [2]:
source_tokens = ["علی", "کتاب", "را", "خواند"]

encoder_states = torch.tensor(
    [
        [1.0, 0.0, 0.0, 0.2],  # علی
        [0.0, 1.0, 0.0, 0.2],  # کتاب
        [0.0, 0.7, 0.3, 0.1],  # را
        [0.2, 0.0, 0.0, 1.0],  # خواند
    ]
)

fixed_context = encoder_states[-1]
print("Fixed context:", fixed_context)
print("Only the final encoder state is directly exposed.")

Fixed context: tensor([0.2000, 0.0000, 0.0000, 1.0000])
Only the final encoder state is directly exposed.


## 2. Luong dot-product attention

We craft decoder states that resemble particular encoder states. This makes the
alignment easy to inspect without training a translation model.

In [3]:
def softmax_torch(x: torch.Tensor, dim: int = -1) -> torch.Tensor:
    return torch.softmax(x, dim=dim)


def luong_dot_attention(
    decoder_state: torch.Tensor,
    encoder_states: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    scores = encoder_states @ decoder_state
    weights = softmax_torch(scores, dim=0)
    context = weights @ encoder_states
    return context, weights


decoder_queries = {
    "Ali": torch.tensor([1.0, 0.0, 0.0, 0.1]),
    "read": torch.tensor([0.1, 0.0, 0.0, 1.0]),
    "book": torch.tensor([0.0, 1.0, 0.1, 0.0]),
}

for target_token, query in decoder_queries.items():
    context, weights = luong_dot_attention(query, encoder_states)
    best_source = source_tokens[int(weights.argmax())]
    print(f"{target_token:>4s} -> {best_source:>5s}; weights={weights.numpy().round(3)}")

 Ali ->   علی; weights=[0.451 0.166 0.164 0.219]
read -> خواند; weights=[0.209 0.189 0.171 0.43 ]
book ->  کتاب; weights=[0.147 0.4   0.305 0.147]


## 3. Bahdanau additive attention as a PyTorch module

Additive attention projects the decoder state and every encoder state, combines
them with `tanh`, and converts the result to one scalar score per source token.

In [4]:
class BahdanauAttention(nn.Module):
    def __init__(self, encoder_dim: int, decoder_dim: int, attention_dim: int):
        super().__init__()
        self.encoder_projection = nn.Linear(encoder_dim, attention_dim, bias=False)
        self.decoder_projection = nn.Linear(decoder_dim, attention_dim, bias=False)
        self.score_projection = nn.Linear(attention_dim, 1, bias=False)

    def forward(
        self,
        decoder_state: torch.Tensor,    # (batch, decoder_dim)
        encoder_states: torch.Tensor,   # (batch, source_len, encoder_dim)
        source_mask: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        projected_encoder = self.encoder_projection(encoder_states)
        projected_decoder = self.decoder_projection(decoder_state).unsqueeze(1)

        scores = self.score_projection(
            torch.tanh(projected_encoder + projected_decoder)
        ).squeeze(-1)

        if source_mask is not None:
            scores = scores.masked_fill(~source_mask, float("-inf"))

        weights = torch.softmax(scores, dim=-1)
        context = torch.bmm(weights.unsqueeze(1), encoder_states).squeeze(1)
        return context, weights


attention = BahdanauAttention(encoder_dim=4, decoder_dim=4, attention_dim=8)
batch_encoder_states = encoder_states.unsqueeze(0)
decoder_state = decoder_queries["read"].unsqueeze(0)

context, weights = attention(decoder_state, batch_encoder_states)
print("Context shape:", tuple(context.shape))
print("Weights shape:", tuple(weights.shape))
print("Row sum:", weights.sum(dim=-1))

Context shape: (1, 4)
Weights shape: (1, 4)
Row sum: tensor([1.], grad_fn=<SumBackward1>)


## 4. Visualize an illustrative alignment matrix

These values are pedagogical alignments, not outputs from the random module above.

In [5]:
target_tokens = ["Ali", "read", "the", "book"]
alignment = np.array(
    [
        [0.82, 0.05, 0.03, 0.10],
        [0.08, 0.10, 0.04, 0.78],
        [0.05, 0.35, 0.50, 0.10],
        [0.04, 0.76, 0.15, 0.05],
    ]
)

fig, ax = plt.subplots()
image = ax.imshow(alignment, aspect="auto")
ax.set_xticks(range(len(source_tokens)), labels=source_tokens)
ax.set_yticks(range(len(target_tokens)), labels=target_tokens)
ax.set_xlabel("Source tokens")
ax.set_ylabel("Target tokens")
ax.set_title("Illustrative recurrent-attention alignment")

for row in range(alignment.shape[0]):
    for column in range(alignment.shape[1]):
        ax.text(column, row, f"{alignment[row, column]:.2f}", ha="center", va="center")

fig.colorbar(image, ax=ax)
plt.show()

/tmp/ipykernel_627/1181597654.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. One attentive decoder step

This module demonstrates the information flow at a single target position:
previous token embedding → recurrent state → attention → vocabulary logits.

In [6]:
class AttentiveDecoderStep(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int,
        hidden_dim: int,
        encoder_dim: int,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.gru_cell = nn.GRUCell(embedding_dim + encoder_dim, hidden_dim)
        self.attention = BahdanauAttention(
            encoder_dim=encoder_dim,
            decoder_dim=hidden_dim,
            attention_dim=hidden_dim,
        )
        self.output = nn.Linear(hidden_dim + encoder_dim, vocab_size)

    def forward(
        self,
        previous_token: torch.Tensor,
        previous_state: torch.Tensor,
        previous_context: torch.Tensor,
        encoder_states: torch.Tensor,
    ):
        embedded = self.embedding(previous_token)
        state_input = torch.cat([embedded, previous_context], dim=-1)
        state = self.gru_cell(state_input, previous_state)
        context, weights = self.attention(state, encoder_states)
        logits = self.output(torch.cat([state, context], dim=-1))
        return logits, state, context, weights


decoder = AttentiveDecoderStep(
    vocab_size=20,
    embedding_dim=8,
    hidden_dim=12,
    encoder_dim=4,
)

batch_size = 1
previous_token = torch.tensor([1])
previous_state = torch.zeros(batch_size, 12)
previous_context = torch.zeros(batch_size, 4)

logits, state, context, weights = decoder(
    previous_token,
    previous_state,
    previous_context,
    batch_encoder_states,
)

print("Vocabulary logits:", tuple(logits.shape))
print("New decoder state:", tuple(state.shape))
print("Context vector:", tuple(context.shape))
print("Attention weights:", tuple(weights.shape))

Vocabulary logits: (1, 20)
New decoder state: (1, 12)
Context vector: (1, 4)
Attention weights: (1, 4)


## Practice

1. Change the crafted query for `book` so it also attends to **را**.
2. Add a padding position and block it with `source_mask`.
3. Replace Luong dot attention with the general score `sᵀWh`.
4. Record one attention row for each decoder step and build a full matrix.
5. Explain why dynamic context removes the single-vector bottleneck but does
   not remove the sequential decoder.